In [1]:
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pickle
from copy import copy
from math import pi, sqrt
%matplotlib qt

parent_dir = os.path.abspath("..")
if parent_dir not in sys.path:
    print(f"appending {parent_dir} to path")
    sys.path.append(parent_dir)

import src.unit_conv as conv

k_B = 1.380649e-23
Mass = 163.929*1.660539*(1e-24/k_B)

appending /home/artur/inzyn/quantum_decaf/view to path


In [2]:
quantum_root = Path(parent_dir).parents[0]
fname = "./70x70x30small/all/time_evol_runs.pickle"
fname = quantum_root / fname

with open(fname, 'rb') as f:
    runs = pickle.load(f)

xr = [runs[0]["r_right"][0][0]]
xl = [runs[0]["r_left"][0][0]]
xrm = [runs[0]["r_right_max"][0][0]]
xlm = [runs[0]["r_left_max"][0][0]]

times = [0]
n_part = runs[0]["n_part"]

e_tot = [runs[0]["energies"]["total"][0]*conv.hartree_to_nK]
e_kin = [runs[0]["energies"]["kinetic"][0]*conv.hartree_to_nK]
e_ext = [runs[0]["energies"]["ext"][0]*conv.hartree_to_nK]
e_contact = [runs[0]["energies"]["contact"][0]*conv.hartree_to_nK]
e_contact_ext = [runs[0]["energies"]["contact_ext"][0]*conv.hartree_to_nK]
e_dd = [runs[0]["energies"]["dipole"][0]*conv.hartree_to_nK]

for i, run in enumerate(runs):
    r_right = run["r_right"][0][1:]
    r_left = run["r_left"][0][1:]
    rm_right = run["r_right_max"][0][1:]
    rm_left = run["r_left_max"][0][1:]

    tot = run["energies"]["total"][1:]*conv.hartree_to_nK
    kin = run["energies"]["kinetic"][1:]*conv.hartree_to_nK
    ext = run["energies"]["ext"][1:]*conv.hartree_to_nK
    contact = run["energies"]["contact"][1:]*conv.hartree_to_nK
    contact_ext = run["energies"]["contact_ext"][1:]*conv.hartree_to_nK
    dd = run["energies"]["dipole"][1:]*conv.hartree_to_nK

    n_steps = run["n_steps"]
    dt = run["timestep"]*conv.jat_to_ms
    timestamps = np.arange(1, n_steps + 1)*dt
    times.extend(timestamps + copy(times[-1]))

    xr.extend(r_right.copy())
    xl.extend(r_left.copy())
    xrm.extend(rm_right.copy())
    xlm.extend(rm_left.copy())

    e_tot.extend(tot.copy())
    e_kin.extend(kin.copy())
    e_ext.extend(ext.copy())
    e_contact.extend(contact.copy())
    e_contact_ext.extend(contact_ext.copy())
    e_dd.extend(dd.copy())

xr = np.array(xr)
xl = np.array(xl)
xrm = np.array(xrm)
xlm = np.array(xlm)

e_tot = np.array(e_tot)
e_kin = np.array(e_kin)
e_ext = np.array(e_ext)
e_contact = np.array(e_contact)
e_contact_ext = np.array(e_contact_ext)
e_dd = np.array(e_dd)

times = np.array(times)

In [3]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(times, xr, "b")
ax.plot(times, xrm, "b--")
ax.plot(times, xl, "r")
ax.plot(times, xlm, "r--")
plt.show()

In [4]:
vr = np.empty_like(xr)
vl = np.empty_like(xl)

vr[0] = (xr[1] - xr[0])/dt
vr[-1] = (xr[-1] - xr[-2])/dt

vl[0] = (xl[1] - xl[0])/dt
vl[-1] = (xl[-1] - xl[-2])/dt

vr[1:-1] = 0.5*(xr[2:] - xr[0:-2])/dt
vl[1:-1] = 0.5*(xl[2:] - xl[0:-2])/dt
tr = 0.5*(n_part*0.5)*Mass*vr**2
tl = 0.5*(n_part*0.5)*Mass*vl**2

In [5]:
def dd_classic(d, h):
    add = 131
    m = 163.929
    rho = (n_part/2/h)
    cdd = 12*pi*add/m*conv.hartree_to_nK/conv.mum_to_atm**3
    print(cdd/4/pi*rho**2*2)
    return cdd/4/pi*rho**2/d*2*(1 - 1/sqrt((h/d)**2 + 1))

In [6]:
xt = np.linspace(2, 10, 101)
yt = np.empty_like(xt)
for i in range(xt.shape[0]):
    yt[i] = dd_classic(xt[i], 5)

fig, ax = plt.subplots()
ax.plot(xt, yt)
plt.show()

181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66319177
181732429.66

In [10]:
dd_eff = e_tot - e_ext - e_contact - e_contact_ext - tr - tl
dd_class = np.empty_like(dd_eff)
dists = np.empty_like(dd_eff)

print("shape is ", dd_eff.shape[0])
for i in range(dd_eff.shape[0]):
    dist = abs(xr[i] - xl[i])
    dists[i] = dist
    dd_class[i] = dd_classic(dist, 3)

fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(times, e_dd - e_dd[0], label="quantum_dd")
ax.plot(times, dd_eff - dd_eff[0], label="eff dd")
ax.plot(times, dd_class - dd_class[0], label="dd classically")
ax2 = ax.twinx()
plt.xlabel("t[ms]")
plt.ylabel("E[nK]")
ax2.plot(times, dists, label="distance between droplect")
ax2.set_ylabel("d [mu m]")
plt.show()

shape is  3001
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.6199771
504812304.619

In [11]:
e_other = e_tot - e_ext
dd_class = np.empty_like(e_other)
dists = np.empty_like(dd_eff)

print("shape is ", dd_eff.shape[0])
for i in range(dd_eff.shape[0]):
    dist = abs(xr[i] - xl[i])
    dists[i] = dist
    dd_class[i] = dd_classic(13*dist, 10)

fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(times, e_other, label="E other")
ax.plot(times, dd_class, label="dd classic")
plt.legend()
plt.show()

shape is  3001
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.41579794
45433107.4157

In [12]:
fig, ax = plt.subplots()
ax.plot(times, e_ext, label="external", color="red")
plt.legend()
ax2 = ax.twinx()
ax2.plot(times, xr - xl, label="dist", color="black")
plt.show()